In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
num_train = 20
num_val = 100

num_inputs = 200 # 입력 데이터 차원 수
batch_size = 5

# weight: [200, 1]
true_weights = torch.full((num_inputs, 1), 0.01)
true_bias = 0.05

# features: [120, 200]
features = torch.randn(num_train + num_val, num_inputs)

# labels: [120, 1]
labels = (
    features @ true_weights
    + true_bias
    + torch.randn(num_train + num_val, 1) * 0.01
)

train_features = features[:num_train]
train_labels = labels[:num_train]

val_features = features[num_train:]
val_labels = labels[num_train:]

train_loader = DataLoader(
    TensorDataset(train_features, train_labels),
    batch_size=batch_size,
    shuffle=True,
)

In [3]:
def train_concise(weight_decay, num_epochs=100, learning_rate=0.01):
    model = nn.Linear(num_inputs, 1)
    
    optimizer = torch.optim.SGD(
        [
            {
                "params": model.weight,
                "weight_decay": weight_decay,
            },
            {
                "params": model.bias,
            },
        ],
        lr=learning_rate,
    )
    
    loss_function = nn.MSELoss()
    
    for epoch in range(num_epochs):
        for batch_features, batch_labels in train_loader:
            predictions = model(batch_features)
            loss = 0.5 * loss_function(predictions, batch_labels)
            
            # 이전 미니배치에서 저장된 gradient를 제거
            optimizer.zero_grad()
            
            # 현재 loss의 gradient를 계산
            loss.backward()
    
            # gradient와 weight decay를 사용해 parameter를 갱신한다.
            optimizer.step()
            
            
    with torch.no_grad():
        train_loss = 0.5 * loss_function(
            model(train_features),
            train_labels,
        ).item()

        val_loss = 0.5 * loss_function(
            model(val_features),
            val_labels,
        ).item()
        
    return model, train_loss, val_loss

In [4]:
model, train_loss, val_loss = train_concise(weight_decay=0)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)
print("Weight L2 norm:", model.weight.norm().item())

Training loss: 3.3751213629473753e-16
Validation loss: 0.1659134030342102
Weight L2 norm: 0.5224104523658752


In [5]:
model, train_loss, val_loss = train_concise(weight_decay=3)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)
print("Weight L2 norm:", model.weight.norm().item())

Training loss: 0.0003979646717198193
Validation loss: 0.008754406124353409
Weight L2 norm: 0.03416914865374565


In [6]:
model, train_loss, val_loss = train_concise(weight_decay=10)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)
print("Weight L2 norm:", model.weight.norm().item())

Training loss: 0.0018100806046277285
Validation loss: 0.008715195581316948
Weight L2 norm: 0.023775646463036537
